In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd

In [2]:
%%time

ddir = 'data/'

df = pd.read_csv(os.path.join(ddir,'chembl_augmented_valid.csv'),usecols=['SMILES'])
df.columns = ['smiles']
display(df)

,smiles
0,CCO
1,OCC
2,C
3,CO
4,OC
...,...
6267871,O(C1C(O)C(n2c(=O)nc(N)cc2)OC1COP(O)(OC1C(O)C(n...
6267872,CC1=CN(C2CC(OP(O)(=O)OCC3OC(C(O)C3OP(O)(=O)OCC...
6267873,C1(O)C(OP(O)(=O)OCC2C(OP(=O)(O)OCC3OC(n4c5[nH]...
6267874,n1(C2CC(OP(O)(=O)OCC3OC(n4c(=O)nc(N)cc4)C(O)C3...


CPU times: user 4.5 s, sys: 456 ms, total: 4.96 s
Wall time: 4.96 s


## 1) Get canonical SMILES ... For all 6 million aug SMILES ...  
Takes 5 minutes in parallel on 14 workers!

In [3]:
%%time
from utilities.rdkit_utils import *
# df['smiles'] = df.smiles.apply(lambda x: get_cansmiles(x))

CPU times: user 60.6 ms, sys: 8.06 ms, total: 68.6 ms
Wall time: 68 ms


In [4]:
# import dask.dataframe as dd
# from dask.multiprocessing import get

import multiprocessing
max_threads = multiprocessing.cpu_count()
print(max_threads)

28


In [9]:
%%time
from pandarallel import pandarallel

pandarallel.initialize(use_memory_fs=False)

df['cansmiles'] = df.smiles.parallel_apply(get_cansmiles)

INFO: Pandarallel will run on 14 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


[23:14:09] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 9 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 64 65 66 67 69 73 74 75 76 79 80 81 82
[23:14:10] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 33 34 35 36 37 38 39 40 41 42 68 69 70 71 72


CPU times: user 2.86 s, sys: 1.3 s, total: 4.16 s
Wall time: 5min 2s


## 2) De-duplicate.

In [11]:
display(df)

,smiles,cansmiles
0,CCO,CCO
1,OCC,CCO
2,C,C
3,CO,CO
4,OC,CO
...,...,...
6267871,O(C1C(O)C(n2c(=O)nc(N)cc2)OC1COP(O)(OC1C(O)C(n...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267872,CC1=CN(C2CC(OP(O)(=O)OCC3OC(C(O)C3OP(O)(=O)OCC...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267873,C1(O)C(OP(O)(=O)OCC2C(OP(=O)(O)OCC3OC(n4c5[nH]...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267874,n1(C2CC(OP(O)(=O)OCC3OC(n4c(=O)nc(N)cc4)C(O)C3...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...


In [21]:
%%time
df_can = df.drop_duplicates(subset='cansmiles',keep='first')[['cansmiles']]
display(df_can)

,cansmiles
0,CCO
2,C
3,CO
5,NCCS
8,NCCN
...,...
6267852,Cc1cn(C2CN(P(=O)(OCC3CN(P(=O)(OCC4CNCC(n5cnc6c...
6267856,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267864,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267868,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...


CPU times: user 1.3 s, sys: 63.4 ms, total: 1.36 s
Wall time: 1.36 s


In [22]:
df_can.columns = ['smiles']
df_can = df_can.dropna()
df_can.to_csv('data/chembl_valid.csv',index=False)

In [23]:
_df = pd.read_csv('data/chembl_valid.csv')
_df

,smiles
0,CCO
1,C
2,CO
3,NCCS
4,NCCN
...,...
1502535,Cc1cn(C2CN(P(=O)(OCC3CN(P(=O)(OCC4CNCC(n5cnc6c...
1502536,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
1502537,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
1502538,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
